In [1]:
# Core
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import seaborn as sns

# Sklearn - Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Sklearn - Models
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Sklearn - Evaluation
from sklearn.metrics import (
    mean_squared_error, r2_score,
    mean_absolute_error, mean_absolute_percentage_error
)
from sklearn.model_selection import cross_val_score, KFold

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('All libraries imported successfully!')

All libraries imported successfully!


In [2]:
df=pd.read_excel("C:\\Users\\HP\\Downloads\\agriculture_ml_project\\merged_crop_enriched_features.xlsx")

In [5]:
df.head()# ── Load dataset ─────────────────────────────────────────────────────────────

print('=' * 60)
print(f'Dataset Shape     : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Years Covered     : {df["Crop_Year"].nunique()} years (2004–2023)')
print(f'Unique Crops      : {df["Crop"].nunique()}')
print(f'Unique Districts  : {df["District_Name"].nunique()}')
print(f'Unique Seasons    : {df["Season"].nunique()}')
print(f'Soil Types        : {df["Soil_Type"].nunique()}')
print(f'Unique Irrigation: {df["Irrigation_Type"].nunique()}')
print('=' * 60)
df.head(10)

Dataset Shape     : 2955 rows × 15 columns
Years Covered     : 19 years (2004–2023)
Unique Crops      : 25
Unique Districts  : 8
Unique Seasons    : 6
Soil Types        : 2
Unique Irrigation: 3


,State_Name,District_Name,Crop_Year,Season,Crop,Area (Hectare),Production (Tonnes/Bales),Soil_Type,Rainfall_mm,Avg_Temperature_C,Humidity_pct,Fertilizer_kg_per_ha,Irrigation_Type,Pest_Disease_Incidence,Yield (Tonne or Bales/Hectare)
0,Tripura,Dhalai,2022 - 2023,Kharif,Arhar/Tur,1459,1221.0000,Red Laterite,288.4000,29.4000,91.5000,58.2000,Canal,Low,0.8400
1,Tripura,Dhalai,2022 - 2023,Kharif,Cotton(lint),160,274.0000,Red Laterite,241.4000,28.1000,79.5000,101.8000,Rainfed,Medium,1.7100
2,Tripura,Dhalai,2022 - 2023,Kharif,Cowpea(Lobia),1315,1093.0000,Red Laterite,300.2000,27.1000,85.4000,41.6000,Canal,Low,0.8300
3,Tripura,Dhalai,2022 - 2023,Rabi,Gram,6,4.0000,Red Laterite,56.4000,16.5000,61.8000,33.2000,Canal,High,0.6700
4,Tripura,Dhalai,2022 - 2023,Kharif,Groundnut,96,136.0000,Red Laterite,193.2000,33.4000,81.7000,62.9000,Canal,Low,1.4200
5,Tripura,Dhalai,2022 - 2023,Rabi,Groundnut,296,457.0000,Red Laterite,68.9000,22.0000,66.3000,64.5000,Rainfed,Medium,1.5400
6,Tripura,Dhalai,2022 - 2023,Kharif,Jute,62,681.0000,Red Laterite,286.6000,33.8000,89.2000,98.7000,Rainfed,Medium,10.9800
7,Tripura,Dhalai,2022 - 2023,Rabi,Khesari,2,2.0000,Red Laterite,60.4000,15.0000,68.9000,37.2000,Rainfed,Low,1.0000
8,Tripura,Dhalai,2022 - 2023,Kharif,Maize,2965,6147.0000,Red Laterite,197.9000,31.1000,90.2000,98.3000,Rainfed,High,2.0700
9,Tripura,Dhalai,2022 - 2023,Rabi,Maize,617,2259.0000,Red Laterite,45.3000,16.6000,62.1000,111.7000,Rainfed,Low,3.6600


In [6]:
# ── Data types & null check ───────────────────────────────────────────────────
print('--- Column Info ---')
df.info()
print('\n--- Missing Values ---')
print(df.isnull().sum())

--- Column Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2955 entries, 0 to 2954
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   State_Name                      2955 non-null   object 
 1   District_Name                   2955 non-null   object 
 2   Crop_Year                       2955 non-null   object 
 3   Season                          2955 non-null   object 
 4   Crop                            2955 non-null   object 
 5   Area (Hectare)                  2955 non-null   int64  
 6   Production (Tonnes/Bales)       2955 non-null   float64
 7   Soil_Type                       2955 non-null   object 
 8   Rainfall_mm                     2955 non-null   float64
 9   Avg_Temperature_C               2955 non-null   float64
 10  Humidity_pct                    2955 non-null   float64
 11  Fertilizer_kg_per_ha            2955 non-null   float64
 12  Irrigation_Typ

In [7]:
# ── Statistical summary ───────────────────────────────────────────────────────
print('--- Numeric Features Summary ---')
df.describe().round(3)

--- Numeric Features Summary ---


,Area (Hectare),Production (Tonnes/Bales),Rainfall_mm,Avg_Temperature_C,Humidity_pct,Fertilizer_kg_per_ha,Yield (Tonne or Bales/Hectare)
count,2955.0000,2955.0000,2955.0000,2955.0000,2955.0000,2955.0000,2955.0000
mean,1899.1800,5228.6710,200.8210,25.2790,75.7810,67.4910,3.8420
std,8373.8130,22905.3020,243.0750,6.0920,10.4830,36.4060,10.4080
min,1.0000,0.7300,20.6000,12.0000,55.3000,24.3000,0.3200
25%,58.0000,63.0000,51.4500,19.8500,66.1000,40.9000,0.7100
50%,176.0000,204.0000,193.1000,27.5000,78.7000,52.6000,0.8800
75%,450.5000,640.0000,259.6000,30.5000,85.1000,86.7000,1.8750
max,102413.0000,282578.0000,1597.9000,35.9000,92.0000,218.6000,66.0100


In [8]:
# ── Categorical value counts ──────────────────────────────────────────────────
cat_cols = ['Season', 'Crop', 'District_Name', 'Soil_Type', 'Irrigation_Type', 'Pest_Disease_Incidence']
for col in cat_cols:
    print(f'\n{col}:')
    print(df[col].value_counts().to_string())


Season:
Season
Kharif        1459
Rabi          1201
Whole Year     120
Summer          63
Autumn          56
Winter          56

Crop:
Crop
Rice                     295
Urad                     239
Moong(Green Gram)        238
Groundnut                228
Maize                    184
Peas & beans (Pulses)    120
Arhar/Tur                120
Jute                     120
Sesamum                  120
Sugarcane                120
Rapeseed &Mustard        120
Mesta                    120
Cotton(lint)             119
Masoor                   119
Other Rabi pulses        112
Gram                     108
Wheat                    106
Other Kharif pulses       84
Small millets             71
Cowpea(Lobia)             64
Soyabean                  55
Jowar                     48
Khesari                   28
Linseed                   12
other oilseeds             5

District_Name:
District_Name
Dhalai           455
South tripura    454
North tripura    453
West tripura     440
Gomati           29